In [ ]:
# Load dataset
import numpy as np
from sklearn import datasets
from sklearn.model_selection import train_test_split

bc = datasets.load_breast_cancer()
X,y = bc.data, bc.target
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.2, random_state=1234
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.25, random_state=1234
)

print("Data shape:",X_train.shape)
print("Label shape:",y_train.shape)

In [ ]:
# Train DL model
from Linear import LinearLayer
from Loss import BCE
from Activation import Sigmoid, ReLU
from Utils import accuracy, normalize_data
from Model import Model

lr = 0.1
bs = len(X_train)
epochs = 100_000
val_freq = 10_000

X_train, X_val, X_test = normalize_data(X_train, X_val, X_test)

lls = [LinearLayer(X_train[0].shape[0], 100), LinearLayer(100, 1)]
sig = Sigmoid()
relu = ReLU()
layers = [LinearLayer(X_train[0].shape[0], 100), ReLU(), LinearLayer(100, 1), Sigmoid()]
model = Model(layers, lr)

for e in range(epochs):
    losses_train, losses_val = [],[]
    preds_train, preds_val = [],[]

    # Training pass
    for i in range(0, len(X_train), bs):
        X = X_train[i:i+bs]
        y = y_train[i:i+bs]

        y_p = model.forward(X)
        loss = BCE.forward(y, y_p)

        preds_train.append(y_p)
        losses_train.append(loss)

        grad = BCE.backward(y, y_p)
        model.backward(grad)
        model.update()
    
    # Validation pass
    if e % val_freq == 0:
        for i in range(0, len(X_val), bs):
            X = X_val[i:i+bs]
            y = y_val[i:i+bs]
            
            y_p = model.forward(X)
            loss = BCE.forward(y, y_p)

            preds_val.append(y_p)
            losses_val.append(loss)

        print(f"Train_loss epoch({e}): {np.average(losses_train):.3f}")
        print(f"Eval_loss epoch({e}): {np.average(losses_val):.3f}")
        print(f"Eval_accuracy: {accuracy(y_val, preds_val):.3f}")
        print("--------------------------------")


In [ ]:
# Evaluate model
losses = []
preds = []
bs = len(X_test)

for i in range(0, len(X_test), bs):
    X = X_test[i:i+bs]
    y = y_test[i:i+bs]
    
    y_p = model.forward(X)
    loss = BCE.forward(y, y_p)

    preds.append(y_p)
    losses.append(loss)

print(f"Test_loss: {np.average(losses):.3f}")
print(f"Test_accuracy: {accuracy(y_test,preds):.3f}")